Group XX | Members: [Student 1], [Student 2] | Question: Q1 | Stage: 2: requirements and acquisition

How this works: we locate the real raw file, otherwise build a clearly labelled stand-in, then document requirements, source provenance, and data dictionary from code.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from datetime import date
import shutil
import pandas as pd
from src.config import RAW_FILE, STANDIN_FILE, COLUMNS, DOCS_DIR
from src.cleaning import sha256_file, to_string_df

DOCS_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILE.parent.mkdir(parents=True, exist_ok=True)

candidates = [
    Path.cwd() / 'maize_yield_rwanda_raw.csv',
    Path.home() / 'Downloads' / 'maize_yield_rwanda_raw.csv',
    RAW_FILE,
]
found = None
for p in candidates:
    if p.exists():
        found = p
        break

standin_used = False
if found is None:
    from tools.make_standin_q1 import main as make_standin
    make_standin(str(STANDIN_FILE))
    use_file = STANDIN_FILE
    standin_used = True
else:
    if found.resolve() != RAW_FILE.resolve():
        shutil.copy2(found, RAW_FILE)
    use_file = RAW_FILE

if use_file == RAW_FILE and RAW_FILE.exists():
    try:
        import os
        os.chmod(RAW_FILE, 0o444)
    except Exception:
        pass

print('Using file:', use_file)
print('Stand-in used:', standin_used)

df_raw = to_string_df(use_file)
print('shape=', df_raw.shape)
display(df_raw.head())
display(df_raw.tail())
print(df_raw.dtypes)
print(df_raw.info())

# data requirements
req_md = f'''# Data Requirements (Q1)

{'**BUILT ON STAND-IN DATA, RE-RUN ON REAL FILE**' if standin_used else ''}

- Target: `yield_kg_per_ha`
- Area: 4 provinces, 14+ districts, 20 cooperatives
- Period: 2023-2025
- Required uses: Questions A-E from stakeholder brief

## Requirement-vs-file check

| Business question | Needed columns | Meets now? | Gaps |
|---|---|---|---|
| A Expected yield | yield_kg_per_ha, farm_size, harvest_weight_kg | Yes | Requires cleaning consistency checks |
| B Fertilizer + seed effect | fertilizer_kg, farm_size, seed_variety | Yes | Must normalize fertilizer per ha |
| C Weather + altitude + irrigation | rainfall_mm, avg_temp_c, altitude_m, irrigation | Yes | Requires realism validation |
| D Location + season differences | province, district, season, planting_date | Yes | Requires label/date harmonization |
| E Low harvest risk (pre-harvest) | pre-harvest fields only | Partial | Must remove leakage columns |
'''
(DOCS_DIR / 'data_requirements.md').write_text(req_md)

# data dictionary
rows = []
meta = {
    'farm_id': ('Farm code', 'string', 'n/a', 'identifier', 'No'),
    'province': ('Province name', 'string', 'n/a', 'Eastern/Southern/Northern/Western', 'No'),
    'district': ('District name', 'string', 'n/a', 'district label', 'No'),
    'cooperative_id': ('Cooperative code', 'string', 'n/a', 'C001..', 'No'),
    'altitude_m': ('Farm altitude', 'numeric', 'meters', 'approx 900-2800', 'No'),
    'season': ('Cropping season', 'categorical', 'n/a', 'A/B/C', 'No'),
    'planting_date': ('Planting date', 'date', 'date', '2023-2025', 'No'),
    'farm_size': ('Farm size value', 'numeric/text', 'mixed', 'with farm_size_unit', 'No'),
    'farm_size_unit': ('Farm size unit', 'categorical', 'ha/m2/acre/are', 'unit variants', 'No'),
    'seed_variety': ('Seed type', 'categorical', 'n/a', 'Local/Improved/Hybrid', 'No'),
    'fertilizer_kg': ('Total fertilizer applied', 'numeric', 'kg', 'whole farm total', 'No'),
    'rainfall_mm': ('Rainfall', 'numeric', 'mm', 'seasonal rainfall', 'No'),
    'avg_temp_c': ('Average temperature', 'numeric', '°C', 'season average', 'No'),
    'soil_type': ('Soil category', 'categorical', 'n/a', 'Clay/Loam/Sandy', 'No'),
    'soil_ph': ('Soil pH', 'numeric', 'pH', 'approx 4-8', 'No'),
    'pest_incidence': ('Pest pressure', 'ordinal', 'n/a', 'Low/Medium/High', 'No'),
    'irrigation': ('Irrigation used', 'categorical', 'n/a', 'Yes/No', 'No'),
    'farmer_age': ('Farmer age', 'numeric', 'years', 'adult expected', 'No'),
    'harvest_weight_kg': ('Harvest weight', 'numeric', 'kg', 'post-harvest leakage risk', 'Yes'),
    'yield_kg_per_ha': ('Yield per hectare (target)', 'numeric', 'kg/ha', 'target', 'Yes'),
}
for c in COLUMNS:
    m = meta.get(c, ('', '', '', '', 'No'))
    rows.append({'column_name': c, 'meaning': m[0], 'data_type': m[1], 'unit': m[2], 'allowed_values_or_notes': m[3], 'known_to_be_post_harvest?': m[4]})
dd = pd.DataFrame(rows)
dd.to_csv(DOCS_DIR / 'data_dictionary.csv', index=False)
(DOCS_DIR / 'data_dictionary.md').write_text('# Data Dictionary\n\n' + dd.to_string(index=False))

# source log
source_md = f'''# Source Log

{'**BUILT ON STAND-IN DATA, RE-RUN ON REAL FILE**' if standin_used else ''}

- Source: MCSU district field forms merged by officers
- File used: `{use_file.name}`
- SHA-256: `{sha256_file(use_file)}`
- Date accessed: {date.today().isoformat()}
- Licence/permission: course use only
- Collected by: district field officers
- Limitations: manual typing, merged rapidly, mixed entry formats
- Ethical issues: no personal identifiers; possible district coverage and reporting bias
'''
(DOCS_DIR / 'source_log.md').write_text(source_md)


Wrote stand-in dataset: /home/runner/work/ML-Learning/ML-Learning/q1_maize_yield/data/raw/maize_yield_rwanda_SYNTHETIC_STANDIN.csv | shape=(1220, 20)
Using file: /home/runner/work/ML-Learning/ML-Learning/q1_maize_yield/data/raw/maize_yield_rwanda_SYNTHETIC_STANDIN.csv
Stand-in used: True
shape= (1220, 20)


,farm_id,province,district,cooperative_id,altitude_m,season,planting_date,farm_size,farm_size_unit,seed_variety,fertilizer_kg,rainfall_mm,avg_temp_c,soil_type,soil_ph,pest_incidence,irrigation,farmer_age,harvest_weight_kg,yield_kg_per_ha
0,F0000,Eastern,Kayonza,C009,1794.7,A,2025-11-24,1.858,ha,Hybrid,114.11,895.8,25.0,Loam,6.72,Medium,No,49,3410.55,1835.21
1,F0001,Eastern,Kayonza,C015,2177.9,A,2024-01-23,1.165,ha,Local,104.11,792.9,23.3,Clay,6.47,Low,No,70,1727.6,1482.86
2,F0002,Southern,Huye,C008,1461.4,A,2023-09-13,1.99,ha,Local,163.59,1085.8,28.4,Sandy,5.28,High,true,65,3008.53,1511.61
3,F0003,Eastern,Bugesera,C012,1240.0,A,2025-01-06,1.247,ha,Local,185.11,782.4,27.7,Loam,5.91,Medium,No,66,1983.92,1590.56
4,F0004,Southern,Nyamagabe,C012,2050.3,B,2024-02-16,1.755,ha,Local,174.12,581.1,23.9,Clay,5.46,Low,No,25,2974.18,1694.36


,farm_id,province,district,cooperative_id,altitude_m,season,planting_date,farm_size,farm_size_unit,seed_variety,fertilizer_kg,rainfall_mm,avg_temp_c,soil_type,soil_ph,pest_incidence,irrigation,farmer_age,harvest_weight_kg,yield_kg_per_ha
1215,f0023,Southern,Nyamagabe,C003,2045.5,A,2024-09-20,1.679,acres,Local,87.72,1232.2,24.3,Loam,6.69,High,No,24,1401.26,2102.23
1216,f0024,Eastern,Gatsibo,C017,1472.1,A,2025-09-13,0.15,ha,Hybrid,7.19,1031.0,27.9,Loam,5.11,High,No,62,324.9,2206.02
1217,f0025,Northern,Musanze,C002,1503.4,B,2025-02-18,17320.932,m2,Improved,249.17,949.2,26.7,Clay,5.82,Low,Yes,26,2258.34,1343.82
1218,f0026,Western,Ngororero,C013,1819.2,A,2025-10-23,7252.149,m2,Local,77.73,1231.9,27.4,Clay,5.56,Medium,No,49,984.09,1396.96
1219,f0027,Southern,Nyanza,C004,1266.2,B,2023-02-06,0.949,ha,Improved,187.44,639.9,28.9,Clay,6.01,Low,No,61,1520.92,1642.0


farm_id              str
province             str
district             str
cooperative_id       str
altitude_m           str
season               str
planting_date        str
farm_size            str
farm_size_unit       str
seed_variety         str
fertilizer_kg        str
rainfall_mm          str
avg_temp_c           str
soil_type            str
soil_ph              str
pest_incidence       str
irrigation           str
farmer_age           str
harvest_weight_kg    str
yield_kg_per_ha      str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 1220 entries, 0 to 1219
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   farm_id            1220 non-null   str  
 1   province           1220 non-null   str  
 2   district           1220 non-null   str  
 3   cooperative_id     1220 non-null   str  
 4   altitude_m         1220 non-null   str  
 5   season             1220 non-null   str  
 6   planting_date    

513